# Deep Learning Project - 2025

### Università di Trento

**Authors:**  
- Antonio N. Bruno (ID: 258035)  
- Edoardo Di Tommaso (ID: 258433)



## Introduction

In recent years, large-scale Vision-Language Models (VLMs) such as CLIP [1] have demonstrated remarkable zero-shot classification performance by aligning images and text in a shared embedding space. This architecture enables flexible, prompt-driven recognition without any task-specific fine-tuning. However, when applied to fine-grained classification tasks, such as distinguishing between species of flowers, birds, or aircraft, the model’s performance tends to degrade, especially in low-data regimes.

This limitation has motivated growing interest in **few-shot adaptation**, where the goal is to adapt a pre-trained VLM to a new classification task using only a small number of labeled examples per class. In this setup, the model is trained on a few samples (shots) from a set of base classes, and then evaluated on both the base and a disjoint set of novel classes. The central challenge is to improve accuracy on base classes without sacrificing generalization to novel ones, a setting often referred to as **base-to-novel generalization**.

Our initial objective was to develop and test a few-shot adaptation method for CLIP on the Oxford Flowers dataset [2]. We experimented with several parameter-efficient fine-tuning (PEFT) approaches, including CoOp [3], CoCoOp [4], and KgCoOp[5]. However, during our investigation we uncovered a surprising finding: a substantial performance bottleneck was caused not by the model’s capacity, but by the **mismatch between dataset class names and CLIP’s training vocabulary**. By carefully aligning class labels with more natural or commonly used names, we significantly improved zero-shot accuracy, surpassing the gains obtained through fine-tuning. This suggests that label engineering can be as impactful as sophisticated adaptation methods.

In this notebook, we present the full workflow of our project: from baseline zero-shot evaluation with original class names, to systematic label alignment experiments, and finally to comparisons with three noteworthy methods from the literature. Alongside code cells, we provide results, tables, figures, and discussion to make the report fully self-contained and reproducible.

### Imports and utilities for data handling

The code cells below import necessary libraries and define utility functions for loading datasets, processing images, and evaluating model performance.

In [ ]:
%pip install openai_clip
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn
%pip install python-dotenv
%pip install google-generativeai

In [ ]:
import clip 
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import ast
import re
from time import sleep
from dotenv import load_dotenv
import json
import google.generativeai as genai

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.
    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include
    Returns:
        subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

## Baseline: CLIP Zero-Shot performance

As a starting point, we evaluated the zero-shot performance of CLIP ViT-B/16 [1] on the Oxford Flowers dataset [2] using the original class names provided in the dataset.
The evaluation was conducted by constructing natural language prompts in the standard CLIP format for Oxford Flowers *“a photo of {class_name}, a type of flower”*, and computing top-1 accuracy separately on base and novel categories.

In [ ]:
# load CLIP. We will use Vit-B/16 as the visual backbone
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device) # preprocess contains CLIP's pre-defined augmentations

# define and inspect base and novel classes
# the class names listed below are the official ones
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
                "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
                "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
                "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
                "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
                "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
                "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup",
                "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
                "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan",
                "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris",
                "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple",
                "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
                "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia", "cyclamen",
                "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss", "foxglove",
                "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
                "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
# zero-shot predictions 

@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # here we apply the standard CLIP template used for Oxford Flowers to all categories
    # and immediately tokenize each sentence
    text_inputs = clip.tokenize(
        [f"a photo of a {CLASS_NAMES[c]}, a type of flower." for c in categories]
    ).to(device)

    # we can encode the text features once as they are shared for all images
    # therefore we do it outside the evaluation loop
    text_features = model.encode_text(text_inputs)
    # and here we normalize them (standard pratice with CLIP)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    # Track per-class predictions and targets
    per_class_correct = torch.zeros(len(categories))
    per_class_total = torch.zeros(len(categories))
    
    for image, target in tqdm(dataloader, desc=label):
        # base categories range from 0 to 50, whil novel ones from 51 to 101
        # therefore we must map categories to the [0, 50], otherwise we will have wrong predictions
        # Map targets in contiguous set starting from zero
        # Labels needs to be .long() in pytorch
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        # forward image through CLIP image encoder
        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # here cosine similarity between image and text features and keep the argmax for every row (every image)
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        # now we check which are correct, and sum them (False == 0, True == 1)
        correct_batch = (predicted_class == target)
        correct_predictions += correct_batch.sum().item()
        
        # Update per-class statistics
        for i in range(len(target)):
            class_idx = target[i].item()
            per_class_total[class_idx] += 1
            if correct_batch[i]:
                per_class_correct[class_idx] += 1

    # and now we compute the accuracy
    accuracy = correct_predictions / len(dataset)
    
    # Compute per-class accuracies
    per_class_accuracies = []
    for i in range(len(categories)):
        if per_class_total[i] > 0:
            class_acc = per_class_correct[i] / per_class_total[i]
            per_class_accuracies.append(class_acc.item())
        else:
            per_class_accuracies.append(0.0)
    
    return accuracy, per_class_accuracies

base_accuracy, base_per_class_acc = eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy, novel_per_class_acc = eval(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")

print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")


Despite being reasonably strong overall, performance varies significantly across classes.
By inspecting per-class accuracies, we stumbled upon a key insight: several classes show notably low accuracy.

In [ ]:
def plot_per_class_accuracies(per_class_accuracies, class_names, categories, title="Per-Class Accuracies"):
    """
    Plot per-class accuracies as a bar chart.
    
    Args:
        per_class_accuracies (list): List of accuracy values for each class
        class_names (list): List of all class names in the dataset
        categories (list): List of class indices being evaluated
        title (str): Title for the plot
    """
    # Get class names for the evaluated categories
    evaluated_class_names = [class_names[i] for i in categories]
    
    # Create figure and axis
    plt.figure(figsize=(15, 8))
    
    # Create bar chart
    bars = plt.bar(range(len(per_class_accuracies)), per_class_accuracies, alpha=0.7)
    
    # Customize the plot
    plt.xlabel('Class Index')
    plt.ylabel('Accuracy')
    plt.title(title)
    plt.xticks(range(len(per_class_accuracies)), [str(categories[i]) for i in range(len(per_class_accuracies))])
    plt.grid(axis='y', alpha=0.3)
    
    # Color bars based on accuracy (red for low, green for high)
    for i, (bar, acc) in enumerate(zip(bars, per_class_accuracies)):
        if acc < 0.25:
            bar.set_color('red')
        elif acc < 0.75:
            bar.set_color('orange')
        else:
            bar.set_color('green')
    
    # Add accuracy values on top of bars
    for i, acc in enumerate(per_class_accuracies):
        plt.text(i, acc + 0.01, f'{acc:.2f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    mean_acc = np.mean(per_class_accuracies)
    std_acc = np.std(per_class_accuracies)
    min_acc = np.min(per_class_accuracies)
    max_acc = np.max(per_class_accuracies)
    
    print(f"\n📊 Per-class accuracy statistics:")
    print(f"   Mean: {mean_acc:.3f}")
    print(f"   Std:  {std_acc:.3f}")
    print(f"   Min:  {min_acc:.3f}")
    print(f"   Max:  {max_acc:.3f}")
    
    # Sort all classes by accuracy (lowest to highest)
    class_acc_pairs = [(idx, acc) for idx, acc in enumerate(per_class_accuracies)]
    sorted_classes = sorted(class_acc_pairs, key=lambda x: x[1])
    
    print(f"\n🔍 All classes sorted by accuracy (lowest to highest):")
    for idx, acc in sorted_classes:
        class_idx = categories[idx]
        print(f"   {class_idx}: {evaluated_class_names[idx]} - {acc:.3f}")

# Plot the per-class accuracies for base and novel classes
plot_per_class_accuracies(base_per_class_acc, CLASS_NAMES, base_classes, 
                         "Per-Class Accuracies - Base Classes (Zero-shot)")

plot_per_class_accuracies(novel_per_class_acc, CLASS_NAMES, novel_classes, 
                         "Per-Class Accuracies - Novel Classes (Zero-shot)")

As we can see, instead of a homogeneous performance across all flower species, certain classes exhibit significantly lower accuracies, dragging down the overall performance. This observation motivated us to investigate the root causes of these discrepancies, leading to our exploration of **label alignment strategies**.

## Investigating Class Name Alignment

Our next step was to systematically explore how modifying labels for poorly performing classes could impact accuracy. Our main intuition was that CLIP’s training data likely contains more common or colloquial names for certain flower species, while the Oxford Flowers dataset uses more technical or less frequent terms. By aligning class names to better match CLIP’s learned vocabulary, we hypothesized that zero-shot performance could be improved.

After many trial and error experiments, we identified 3 specific strategies for label alignment that yielded significant accuracy improvements:

1. **Synonym Replacement**: replacing the original class name with a more common synonym or alternative name.
2. **Formatting Tweaks**: small change such as adding or removing dashes or slightly adjusting the wording to better match common usage.
3. **Corrections**: in a few cases, we found that the original class name was in fact not matching the flower species depicted in the images, so we corrected it to the appropriate name.

Let's now see a few examples for each strategy.

### 1. Synonym Replacement

Let's consider for instance base class 33, originally named *"mexican aster"*, with a zero-shot accuracy of 10%.

With a quick google search, we can find that this flower is also commonly referred to as *"garden cosmos"*.

By replacing the original label with this synonym, we observed a significant increase in accuracy.

In [ ]:
# replace the class name with its synonym
CLASS_NAMES[33] = "garden cosmos"

# re-evaluate CLIP after the change
base_accuracy, base_per_class_acc = eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes" )


# print and plot the results
print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
plot_per_class_accuracies(base_per_class_acc, CLASS_NAMES, base_classes, "Per-Class Accuracies - Base Classes (Zero-shot)")

# revert the class name change
CLASS_NAMES[33] = "mexican aster"

As we can see, class 33's accuracy improved from 10% to 100% simply by changing the label to a more common synonym. This confirms our hypothesis that better alignment of class names with CLIP's training vocabulary can lead to substantial performance gains.

### 2. Formatting Tweaks

Another effective strategy we found was to make small formatting adjustments to class names.

Let's consider base class 32, originally named *"love in the mist"*, with a zero-shot accuracy of 0%.

Again, with a quick search we can find that this flower is commonly referred to as *"love-in-a-mist"* (with dashes).

By applying this formatting tweak, we observed a remarkable improvement in accuracy.

In [ ]:
# replace the class name with its new name
CLASS_NAMES[32] = "love-in-a-mist"

# re-evaluate CLIP after the change
base_accuracy, base_per_class_acc = eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes" )


# print and plot the results
print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
plot_per_class_accuracies(base_per_class_acc, CLASS_NAMES, base_classes, "Per-Class Accuracies - Base Classes (Zero-shot)")

# revert the class name change
CLASS_NAMES[32] = "love in the mist"

We can see how a very simple formatting change led to a dramatic increase in accuracy from 0% to 100%. This further illustrates the sensitivity of CLIP's zero-shot performance to the exact wording and formatting of class names.

### 3. Corrections

To our surprise, by searching by image some classes' sample, we realized the original class name and the sample images led to different flowers, and did not correspond. By correcting these mislabelings, we were able to significantly boost accuracy for these classes.

The most striking example is novel class 88, originally named *"watercress"*, with a zero-shot accuracy of 0%.

At first we tried to find synonyms or formatting tweaks, but accuracy remained at 0%.

Then, by inspecting the images, we realized that research by image associated them with another flower species: *"nasturtium"*. By changing the label to this name, we observed a dramatic increase in accuracy, suggesting that the original label was not aligned with the learned knowledge of the network, which tends to reflect the most popular associations made online, regardless if they are right or wrong.

In [ ]:
# replace the class name with its new name
CLASS_NAMES[88] = "nasturtium"

# re-evaluate CLIP after the change
novel_accuracy, novel_per_class_acc = eval(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes" )


# print and plot the results
print()
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")
plot_per_class_accuracies(novel_per_class_acc, CLASS_NAMES, novel_classes, "Per-Class Accuracies - Novel Classes (Zero-shot)")

# revert the class name change
CLASS_NAMES[88] = "watercress"

As we can see, the accuracy for class 88 jumped from 0% to almost 100% after correcting the label. This highlights the importance of accurate labeling and how mislabelings can severely impact model performance. Moreover, since this class is a novel one, there would be no way to improve its accuracy in a base-to-novel generalization setting without correcting the label.

## AKA CLIP

Given the surprising effectiveness of simple label changes, we decided to modify CLIP zero shot to enable the model to use multiple names/aliases for each class, while still selecting the most similar one for each sample, and trace it back to the original class. By allowing to use multiple names alltogether, we can observe and select the best performing aliases, or alternatively, by automating the collection/generation of synonims, automate the classification task, by removing human intervention.

In [ ]:
AKA_CLASS_NAMES = [['pink primrose', 'pink evening primrose'], ['hard-leaved pocket orchid', 'silver slipper orchid'],
               ['canterbury bells', 'campanula medium'], ['sweet pea', 'lathyrus odoratus'], ['english marigold', 'calendula'],
               ['tiger lily'], ['moon orchid'], ['bird of paradise'], ['monkshood'], ['globe thistle'], ['snapdragon'],
               ["colt's foot", 'coltsfoot'], ['king protea'], ['spear thistle'], ['yellow iris'], ['globe-flower', 'trollius'],
               ['purple coneflower'], ['peruvian lily'], ['balloon flower', 'chinese bellflower'], ['giant white arum lily'],
               ['fire lily'], ['pincushion flower'], ['fritillary'], ['red ginger'], ['grape hyacinth'], ['corn poppy'],
               ['prince of wales feathers', 'amaranthus hypochondriacus'], ['stemless gentian'], ['artichoke', 'globe artichoke'],
               ['sweet william'], ['carnation'], ['garden phlox'], ['love in the mist', 'love-in-a-mist'],
               ['mexican aster', 'garden cosmos'], ['alpine sea holly'], ['ruby-lipped cattleya'],
               ['cape flower', 'japanese spider lily'], ['great masterwort', 'astrantia major'],
               ['siam tulip', 'curcuma alismatifolia'], ['lenten rose'], ['barbeton daisy', 'gerbera jamesonii'], ['daffodil'],
               ['sword lily', 'marsh gladiolus'], ['poinsettia'], ['bolero deep blue', 'eustoma exaltatum'],
               ['wallflower', 'erysimum'], ['marigold'], ['buttercup'], ['oxeye daisy'], ['common dandelion'], ['petunia'],
               ['wild pansy'], ['primula'], ['sunflower'], ['lilac hibiscus'], ['bishop of llandaff', 'dahlia bishop of llandaff'],
               ['gaura'], ['geranium'], ['orange dahlia'], ['pink-yellow dahlia', 'pink and yellow dahlia'], ['cautleya spicata'],
               ['japanese anemone'], ['black-eyed susan'], ['silverbush', 'shrubby bindweed'], ['californian poppy'],
               ['osteospermum'], ['spring crocus'], ['bearded iris'], ['windflower', 'wood anemone'], ['tree poppy'], ['gazania'],
               ['azalea'], ['water lily'], ['rose'], ['thorn apple', 'jimsonweed'], ['morning glory', 'morning-glory'],
               ['passion flower'], ['lotus'], ['toad lily'], ['anthurium'], ['frangipani'], ['clematis'], ['hibiscus'],
               ['columbine'], ['desert-rose'], ['tree mallow'], ['magnolia'], ['cyclamen'], ['watercress', 'nasturtium'],
               ['canna lily'], ['hippeastrum'], ['bee balm'], ['ball moss', 'tillandsia cyanea'], ['foxglove'], ['bougainvillea'],
               ['camellia'], ['mallow', 'abutilon'], ['mexican petunia', 'ruellia'], ['bromelia'], ['blanket flower'],
               ['trumpet creeper', 'campsis radicans'], ['blackberry lily']]

In [ ]:
@torch.no_grad() # we don't want gradients
def aka_eval(model: nn.Module,
         dataset: torch.utils.data.Dataset,
         categories: list[int],
         batch_size: int,
         device: str,
         class_names=AKA_CLASS_NAMES,
         label=""):
    model.eval()

    # Remap labels into a contiguous set starting from zero
    true_label_to_index = {cat: idx for idx, cat in enumerate(categories)}
    log = pd.DataFrame(columns=["true_label", "predicted_label", "correct"])
    prompts = []
    map_alias_index_to_index_label = []
    map_alias_index_to_alias_label = []
    for c in categories:
        for alias in class_names[c]:
            prompts.append(f"a photo of a {alias}, a type of flower.")
            map_alias_index_to_index_label.append(true_label_to_index[c])
            map_alias_index_to_alias_label.append(alias)
    text_inputs = clip.tokenize(prompts).to(device)
    text_features = model.encode_text(text_inputs)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    correct_predictions = 0
    for image, target in tqdm(dataloader, desc=label):
        true_labels = [t.item() for t in target]
        target = torch.Tensor([true_label_to_index[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        predicted_labels = [map_alias_index_to_alias_label[p.item()] for p in predicted_class]
        predicted_class = torch.Tensor([map_alias_index_to_index_label[p.item()] for p in predicted_class]).long().to(device)
        prediction_correctness = (predicted_class == target).tolist()
        batch_data = []
        for i, correct in enumerate(prediction_correctness):
            batch_data.append({"true_label": true_labels[i],
                              "predicted_label": predicted_labels[i],
                              "correct": correct})
        
        batch_df = pd.DataFrame(batch_data)
        log = pd.concat([log, batch_df], ignore_index=True)
        correct_predictions += sum(prediction_correctness)
    accuracy = correct_predictions / len(dataset)
    return accuracy, log

base_accuracy, base_log = aka_eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy, novel_log = aka_eval(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")
print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")
print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")

In [ ]:
def compute_alias_metrics(base_log, novel_log, class_names=AKA_CLASS_NAMES):
    map_alias_to_label_index = {}
    for index in range(len(class_names)):
        for alias in class_names[index]:
            map_alias_to_label_index[alias] = index
    base_classes_sample_counts = base_log['true_label'].value_counts().sort_index()
    novel_classes_sample_counts = novel_log['true_label'].value_counts().sort_index()

    # Concatenate base and novel counts
    all_classes_sample_counts = pd.concat([base_classes_sample_counts, novel_classes_sample_counts]).sort_index()

    # Merge base and novel logs
    combined_log = pd.concat([base_log, novel_log], ignore_index=True)

    # Compute correct predictions for each alias
    alias_correct_predictions = combined_log[combined_log['correct'] == True]['predicted_label'].value_counts()

    # Create metrics dictionary for each alias
    alias_metrics = {}
    for alias in map_alias_to_label_index.keys():
        class_index = map_alias_to_label_index[alias]
        
        # Get correct predictions for this alias
        correct_preds = alias_correct_predictions.get(alias, 0)
        
        # Get total samples for this class
        total_samples = all_classes_sample_counts.get(class_index, 0)
        
        # Get total predictions for this alias (correct + incorrect)
        total_predictions = combined_log[combined_log['predicted_label'] == alias].shape[0]
        
        # Calculate precision (correct predictions / total predictions for this alias)
        precision = correct_preds / total_predictions if total_predictions > 0 else 0.0
        recall = correct_preds / total_samples if total_samples > 0 else 0.0
        
        alias_metrics[alias] = {
            'correct_predictions': correct_preds,
            'total_class_samples': total_samples,
            'total_predictions': total_predictions,
            'precision': precision,
            'recall': recall
        }
    alias_metrics_df = pd.DataFrame.from_dict(alias_metrics, orient='index')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(alias_metrics_df)

compute_alias_metrics(base_log, novel_log, class_names=AKA_CLASS_NAMES)

In [ ]:
def compute_class_accuracies(base_log, novel_log, class_names=AKA_CLASS_NAMES):
    # Count correct predictions for each class
    base_correct_predictions_counts = base_log[base_log['correct'] == True]['true_label'].value_counts().sort_index()
    novel_correct_predictions_counts = novel_log[novel_log['correct'] == True]['true_label'].value_counts().sort_index()
    base_classes_sample_counts = base_log['true_label'].value_counts().sort_index()
    novel_classes_sample_counts = novel_log['true_label'].value_counts().sort_index()
    all_classes_sample_counts = pd.concat([base_classes_sample_counts, novel_classes_sample_counts]).sort_index()

    # Concatenate base and novel correct predictions
    all_classes_correct_predictions_counts = pd.concat([base_correct_predictions_counts, novel_correct_predictions_counts]).sort_index()

    # Calculate accuracy for each class
    class_accuracies = {}
    for class_id in range(len(class_names)):
        correct_preds = all_classes_correct_predictions_counts.get(class_id, 0)
        total_samples = all_classes_sample_counts.get(class_id, 0)
        recall = correct_preds / total_samples if total_samples > 0 else 0.0
        
        class_accuracies[class_id] = {
            'class_name': class_names[class_id][0],  # Primary name for the class
            'all_aliases': class_names[class_id],    # All aliases for this class
            'correct_predictions': correct_preds,
            'total_samples': total_samples,
            'recall': recall,
        }

    # Convert to DataFrame
    class_accuracies_df = pd.DataFrame.from_dict(class_accuracies, orient='index')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(class_accuracies_df)

compute_class_accuracies(base_log, novel_log, class_names=AKA_CLASS_NAMES)

In [ ]:
def class_confusion_matrix(log, selected_classes, label="base", class_names=AKA_CLASS_NAMES):
    map_alias_to_label_index = {}
    for index in range(len(class_names)):
        for alias in class_names[index]:
            map_alias_to_label_index[alias] = index
    # Convert predicted aliases to class indices for log
    log_with_pred_indices = log.copy()
    log_with_pred_indices['predicted_label'] = log_with_pred_indices['predicted_label'].map(map_alias_to_label_index)
    log_with_pred_indices['true_label'] = log_with_pred_indices['true_label'].astype(int)
    log_with_pred_indices['predicted_label'] = log_with_pred_indices['predicted_label'].astype(int)

    # Get true and predicted class indices
    y_true = log_with_pred_indices['true_label'].values
    y_pred = log_with_pred_indices['predicted_label'].values

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=selected_classes)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    cm_percent = np.nan_to_num(cm_percent)


    # Plot confusion matrix
    plt.figure(figsize=(20, 16))
    sns.heatmap(cm_percent, 
                xticklabels=selected_classes, 
                yticklabels=selected_classes,
                annot=True, 
                fmt='.2f', 
                cmap='Blues',
                cbar_kws={'label': 'Percentage of Predictions'})

    plt.title(f'Confusion Matrix for {label} Classes (CLIP Zero-shot)', fontsize=16, pad=20)
    plt.xlabel('Predicted Class', fontsize=14)
    plt.ylabel('True Class', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Print some statistics about the confusion matrix
    print(f"📊 Confusion Matrix Statistics:")
    print(f"Matrix shape: {cm.shape}")
    print(f"Total predictions: {cm.sum()}")
    print(f"Correct predictions (diagonal): {np.trace(cm)}")
    print(f"Accuracy: {np.trace(cm) / cm.sum():.4f}")

    # Find most confused classes (highest off-diagonal values)
    cm_percent_off_diagonal = cm_percent.copy()
    np.fill_diagonal(cm_percent_off_diagonal, 0)
    max_confusion_idx = np.unravel_index(np.argmax(cm_percent_off_diagonal), cm_percent_off_diagonal.shape)
    max_confusion_count = cm_percent_off_diagonal[max_confusion_idx]
    max_confusion_percent = cm_percent_off_diagonal[max_confusion_idx]

    true_class_idx = selected_classes[max_confusion_idx[0]]
    pred_class_idx = selected_classes[max_confusion_idx[1]]

    print(f"\n🔀 Most confused pair:")
    print(f"True class: {true_class_idx} (class {true_class_idx})")
    print(f"Predicted as: {pred_class_idx} (class {pred_class_idx})")
    print(f"Confusion count: {max_confusion_count}")
    print(f"Confusion percentage: {max_confusion_percent:.1f}%")

class_confusion_matrix(base_log, base_classes, class_names=AKA_CLASS_NAMES)

In [ ]:
class_confusion_matrix(novel_log, novel_classes, label="novel", class_names=AKA_CLASS_NAMES)

As we can see from the results, there's a significant improvement in performance compared to the zero-shot model with original names; some classes are still underperforming, due to the presence of some classnames that disrupt the correct classification, mostly due to being associated with the wrong classes, and belonging to the third case of label change.

### LLM-generated Aliases
Lastly, we tried to generate a list of synonims for each class through the use of a large language model, to fully automate the pipeline.

In [ ]:
generate_names = True
try:
    load_dotenv()
    api_key = os.getenv("GOOGLE_API_KEY")
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-2.5-flash")
except Exception as e:
    print("⚠️ Could not configure Google Gemini API. Stored names from previous generation will be used.")
    generate_names = False
    print(e)

In [ ]:
aliases_dict = {}

# Ask a question
prompt = """You are a concise, factual assistant that returns only lists of aliases (no explanations).

Instructions:
1) Given the canonical Flower/Plant name after the colon, return ONLY its common aliases or other names typically used to identify that specific flower.
2) Output must be a valid Python list literal (i.e. start with '[' and end with ']'). Example format exactly:
   Flower/Plant name: Chrysanthemums
   List of alternative names: ['Mums', 'Chrysanths']
3) Do NOT include any text besides the Python list for the requested flower (no labels, no punctuation outside the list, no trailing whitespace, no comments).
4) If there are no known aliases, return an empty list: []
5) Preserve capitalization as commonly used for each alias; include multi-word aliases as single strings.
6) If an alias is identical to the input name, do NOT repeat it.
7) If you are unsure about a name, prefer returning fewer aliases rather than guessing. Do not hallucinate.

Few-shot examples (must be followed exactly):
Flower/Plant name: Chrysanthemums
List of alternative names: ['Mums', 'Chrysanths']

Flower/Plant name: Bellis perennis
List of alternative names: ['English Daisy', 'Lawn Daisy', 'Common Daisy']

Now complete for the requested flower:
Flower/Plant name: {} 
List of alternative names:"""


In [ ]:
if generate_names:
    for flower in CLASS_NAMES:
        formatted_prompt = prompt.format(flower)
        max_retries = 3
        retry_count = 0
        
        while retry_count < max_retries:
            try:
                response = model.generate_content(formatted_prompt)
                print(response.text)
                aliases = response.text.strip()
                
                # Use regex to extract only the list content between brackets
                list_pattern = r'\[.*?\]'
                match = re.search(list_pattern, aliases)
                
                if match:
                    list_string = match.group(0)
                    aliases_list = ast.literal_eval(list_string)
                    aliases_dict[flower] = aliases_list
                    break  # Success, exit retry loop
                else:
                    raise ValueError("No list pattern found in response")
            except ValueError as ve:
                continue  # Retry on parsing errors
            except Exception as e:
                print(e)
                retry_count += 1
                print(f"Error processing {flower} (attempt {retry_count}/{max_retries}): {e}")
                if retry_count < max_retries:
                    print("Waiting 60 seconds before retry...")
                    sleep(60)
                else:
                    print(f"Failed to process {flower} after {max_retries} attempts")
    GENERATED_AKA_CLASS_NAMES = []
    for key, value in aliases_dict.items():
        GENERATED_AKA_CLASS_NAMES.append([key] + value)
else:
    GENERATED_AKA_CLASS_NAMES = [["pink primrose","Evening primrose","Mexican primrose","Showy primrose","Oenothera speciosa","Pinkladies","Pink lady"],
                                 ["hard-leaved pocket orchid","Grass-leaved orchid","Japanese pocket orchid"],["canterbury bells"],
                                 ["sweet pea","lathyrus odoratus"],["english marigold","Pot Marigold","Calendula","Garden Marigold","Common Marigold"],
                                 ["tiger lily","Lilium lancifolium","Lilium tigrinum","Devil Lily","Leopard Lily","Outhouse Lily"],
                                 ["moon orchid","Phalaenopsis orchid","Phal orchid","Moth orchid"],
                                 ["bird of paradise","Crane Flower","Strelitzia","Orange Bird of Paradise"],
                                 ["monkshood","Aconitum","Wolfsbane","Devil's Helmet","Queen of Poisons","Blue Rocket","Helmet Flower"],
                                 ["globe thistle","Echinops","echinops thistle"],["snapdragon","Dragon flower","Dog flower"],
                                 ["colt's foot","Coughwort","Foalswort","Horsehoof","Clayweed","Ass's foot","Donnhove","Tussilago"],
                                 ["king protea","Giant Protea","Honeypot","King Sugar Bush"],["spear thistle","Bull Thistle","Common Thistle","Lance Thistle"],
                                 ["yellow iris","Yellow Flag","Yellow Flag Iris","Water Flag"],["globe-flower","Trollius","Globeflower"],
                                 ["purple coneflower","Echinacea","Eastern purple coneflower"],
                                 ["peruvian lily","Alstroemeria","Lily of the Incas","Princess Lily"],
                                 ["balloon flower","Platycodon","Chinese Bellflower","Japanese Bellflower","Korean Bellflower"],
                                 ["giant white arum lily","Calla Lily","White Arum Lily","Pig Lily","Trumpet Lily","Lily of the Nile","Ethiopian Lily"],
                                 ["fire lily","Orange Lily","St. John's Lily"],["pincushion flower","Scabious"],
                                 ["fritillary","Fritillaria","Snake's Head Fritillary","Chequered Lily","Crown Imperial","Guinea-hen flower"],
                                 ["red ginger","Ostrich Plume","Jungle Queen","Jungle King","Tahitian Ginger"],["grape hyacinth","Muscari"],
                                 ["corn poppy","common poppy","field poppy","Flanders poppy","Shirley poppy"],
                                 ["prince of wales feathers","Feather Celosia","Plumed Celosia"],
                                 ["stemless gentian","Trumpet Gentian","Alpine Gentian","Acaulis Gentian"],
                                 ["artichoke","Globe Artichoke","French Artichoke","Cynara scolymus","Thistle Artichoke"],
                                 ["sweet william","Dianthus barbatus","Bearded Pinks"],["carnation","Clove Pink","Dianthus caryophyllus","Grenadine"],
                                 ["garden phlox","Summer Phlox","Tall Garden Phlox","Perennial Phlox","Border Phlox","Phlox paniculata"],
                                 ["love in the mist","Nigella","Nigella damascena","Devil in the Bush","Lady-in-the-Bower"],
                                 ["mexican aster","Cosmos","Cosmea","Garden Cosmos"],["alpine sea holly","Alpine Thistle","Alpine Eryngo"],
                                 ["ruby-lipped cattleya","Cattleya","Cattleya orchid","Standard Cattleya","Autumn Cattleya"],
                                 ["cape flower","Cape primrose","Streptocarpus"],["great masterwort","masterwort","Astrantia major"],
                                 ["siam tulip","Curcuma alismatifolia","Summer Tulip","Tulip Ginger","Ginger Lily","Thai Tulip"],
                                 ["lenten rose","Christmas Rose","Hellebore","Winter Rose"],
                                 ["barbeton daisy","Gerbera Daisy","Transvaal Daisy","Veldt Daisy","Gerbera"],["daffodil","Narcissus","Jonquil","Lent Lily"],
                                 ["sword lily","Gladiolus","Gladioli"],["poinsettia","Christmas flower","Christmas star","Nochebuena","Mexican flameleaf"],
                                 ["bolero deep blue","Petunia 'Bolero Deep Blue'","Bolero Deep Blue Petunia","Bolero Petunia"],
                                 ["wallflower","Gillyflower","Erysimum"],["marigold","calendula","pot marigold","African marigold","French marigold","Tagetes"],
                                 ["buttercup","Crowfoot","Goldcup","Butter Rose","Kingcup"],
                                 ["oxeye daisy","Dog daisy","Moon daisy","Moonpenny","Field daisy","Marguerite"],
                                 ["common dandelion","Dandelion","Lion's Tooth","Blowball","Puffball","Piss-a-bed"],
                                 ["petunia","Wave Petunia","Supertunia","Calibrachoa"],
                                 ["wild pansy","Heartsease","Johnny Jump Up","Viola tricolor","Love-in-idleness","Tickle-my-fancy","Kiss-me-at-the-garden-gate","Three Faces in a Hood"],
                                 ["primula","Primrose"],["sunflower","Common Sunflower","Helianthus annuus"],["pelargonium","Geranium","Storksbill"],
                                 ["bishop of llandaff","Dahlia 'Bishop of Llandaff'","Bishop of Llandaff Dahlia"],
                                 ["gaura","Whirling Butterflies","Lindheimer's Beeblossom","Beeblossom"],["geranium","cranesbill","pelargonium"],
                                 ["orange dahlia",""],["pink-yellow dahlia?"],["cautleya spicata","Himalayan Ginger","Spiked Cautleya","Orange Ginger"],
                                 ["japanese anemone","Windflower","Anemone hupehensis","Thimbleweed"],
                                 ["black-eyed susan","Rudbeckia hirta","gloriosa daisy","yellow ox-eye daisy","brown-eyed susan","brown betty","coneflower","golden Jerusalem"],
                                 ["silverbush","Shrubby Bindweed","Mediterranean Silverbush"],
                                 ["californian poppy","Golden Poppy","California Poppy","Copa de Oro","Eschscholzia","State Flower of California"],
                                 ["osteospermum","African daisy","Cape daisy","Spoon daisy","Dimorphotheca"],["spring crocus","Dutch Crocus","Crocus vernus"],
                                 ["bearded iris","German Iris","Flag Iris"],["windflower","Anemone"],
                                 ["tree poppy","Dendromecon","Matilija poppy","California tree poppy","bush poppy"],["gazania","Treasure Flower","African Daisy"],
                                 ["azalea","Rhododendron","Shrub Azalea"],["water lily","Water Nymph","Lotus","Nymphaea"],["rose","Queen of Flowers"],
                                 ["thorn apple","Datura","Jimsonweed","Devil's Snare","Devil's Trumpet","Hell's Bells","Stinkweed","Loco Weed","Pricklyburr","Moonflower"],
                                 ["morning glory","bindweed","Heavenly Blue","Grandpa Ott","Ipomoea purpurea"],
                                 ["passion flower","Passiflora","Maypop","Purple Passionflower","Christ's Crown"],["lotus","Sacred Lotus","Indian Lotus"],
                                 ["toad lily","Tricyrtis","Japanese toad lily"],["anthurium","Flamingo Flower","Painter's Palette","Tailflower","Laceleaf"],
                                 ["frangipani","Plumeria","Lei flower","Temple flower","Pagoda flower"],
                                 ["clematis","Leather Flower","Vining Clematis","Virgin's Bower"],["hibiscus","Rose Mallow","China Rose","Shoeblackplant"],
                                 ["columbine","Granny’s Bonnet","Aquilegia","European Crowfoot"],
                                 ["desert-rose","Adenium","Sabi Star","Kudu Lily","Mock Azalea","Impala Lily","Star of Lundi"],
                                 ["tree mallow","Malva arborea","Lavatera arborea","sea mallow"],["magnolia","Mag","Magnolias"],
                                 ["cyclamen","Persian violet","Sowbread","Alpine violet"],["watercress","Water rocket"],
                                 ["canna lily","Canna","Indian Shot","Queensland Arrowroot"],["hippeastrum","Amaryllis","Christmas Amaryllis","Dutch Amaryllis"],
                                 ["bee balm","Monarda","Wild bergamot","Oswego tea","Bergamot"],["ball moss","Small ball moss","Bunch moss"],
                                 ["foxglove","Digitalis","Lady’s Glove","Fairy Thimbles","Dead Man’s Bells","Bloody Bells","Witches’ Gloves","Lion’s Mouth"],
                                 ["bougainvillea","Paperflower","Trinitaria"],["camellia","Winter Rose","Japanese Camellia","Common Camellia"],
                                 ["mallow","Malva","Cheeses","High Mallow","Common Mallow"],
                                 ["mexican petunia","Ruellia","Mexican bluebell","Britton's wild petunia","Lavender bells","Florida petunia"],
                                 ["bromelia","Bromeliad","Air Plant"],["blanket flower","Gaillardia","Indian Blanket","Firewheel"],
                                 ["trumpet creeper","Trumpet Vine","Cow-itch Vine","Devil's Shoestring","Hellflower"],
                                 ["blackberry lily","Leopard flower","Leopard lily","Belamcanda","Iris domestica","Apostle plant"]]

In [ ]:
base_llm_generated_accuracy, base_llm_generated_log = aka_eval(model=clip_model, dataset=test_base, class_names=GENERATED_AKA_CLASS_NAMES, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_llm_generated_accuracy, novel_llm_generated_log = aka_eval(model=clip_model, dataset=test_novel, class_names=GENERATED_AKA_CLASS_NAMES, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")
print()
print(f"🔍 Base classes accuracy: {base_llm_generated_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_llm_generated_accuracy*100:.2f}%")
print(f"🔍 Harmonic Mean: {harmonic_mean(base_llm_generated_accuracy, novel_llm_generated_accuracy)*100:.2f}%")

In [ ]:
compute_alias_metrics(base_llm_generated_log, novel_llm_generated_log, class_names=GENERATED_AKA_CLASS_NAMES)

In [ ]:
compute_class_accuracies(base_llm_generated_log, novel_llm_generated_log, class_names=GENERATED_AKA_CLASS_NAMES)

In [ ]:
class_confusion_matrix(base_llm_generated_log, base_classes, label="base", class_names=GENERATED_AKA_CLASS_NAMES)

In [ ]:
class_confusion_matrix(novel_llm_generated_log, novel_classes, label="novel", class_names=GENERATED_AKA_CLASS_NAMES)

As we can see from the achieved results, simply using an LLM to generate a list of possible aliases for each class and using them concurrently doesn't help achieving satisfactory results. Some names that were initially wrong, create even more confusion in the classification task, that when added with some erorrs of the LLM and misalignment of knowledge between the two model leads to poor results.

## References

[1] Radford et Al. Learning transferable visual models from natural language supervision. In ICML, 2021. https://arxiv.org/abs/2103.00020

[2] Nilsback, M.-E. and Zisserman, A. Automated flower classification over a large number of classes. In Indian Conference on Computer Vision, Graphics and Image Processing, 2008. https://ieeexplore.ieee.org/document/4756141

[3] Zhou et Al. Learning to prompt for vision-language models. In ICCV, 2021. https://arxiv.org/abs/2109.01134

[4] Zhou et Al. Conditional prompt learning for vision-language models. In NeurIPS, 2022. https://arxiv.org/abs/2203.05557

[5] Yao et Al. Visual-Language Prompt Tuning with Knowledge-guided Context Optimization In CVPR, 2023. https://arxiv.org/abs/2303.13283